# 🧊 NumPy Shape Manipulation: Reshaping, Stacking & Splitting

## 1. Reshaping & Transposing (The "Metadata" Hack)
**Memory Level Reality:** Operations like `reshape`, `transpose` (`.T`), and `ravel` **do not physically move or copy data in RAM.** 
Instead, NumPy simply changes the *metadata* (Shape and Strides). Since it’s just reading the same flat C-array from a different angle, these operations are incredibly fast ($O(1)$ time complexity) and return a **View**.
*   `reshape`: Changes the dimensions (e.g., 1D to 2D).
*   `Transpose (.T)`: Swaps the axes (Rows become Columns).
*   `ravel`: Flattens a multi-dimensional array back into 1D (usually returns a View, unlike `flatten()` which forces a Copy).

In [2]:
import numpy as np

In [3]:
# --- 1. RESHAPING & TRANSPOSING ---
a = np.arange(10)
a_reshaped = a.reshape(5, 2)
print("Reshaped (5x2):\n", a_reshaped)

a2 = np.arange(12).reshape(3, 4)
print("\nOriginal a2 (3x4):\n", a2)

# Transpose (Rows <-> Columns)
print("\nTranspose using np.transpose:\n", np.transpose(a2))
print("Transpose using .T (Preferred):\n", a2.T)

# Ravel (Flattening to 1D View)
a3 = np.arange(8).reshape(2, 2, 2)
print("\nRavel a2:\n", a2.ravel())
print("Ravel a3:\n", a3.ravel())

Reshaped (5x2):
 [[0 1]
 [2 3]
 [4 5]
 [6 7]
 [8 9]]

Original a2 (3x4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Transpose using np.transpose:
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
Transpose using .T (Preferred):
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]

Ravel a2:
 [ 0  1  2  3  4  5  6  7  8  9 10 11]
Ravel a3:
 [0 1 2 3 4 5 6 7]


## 2. Stacking (The RAM Consumer)
**Memory Level Reality:** Unlike reshaping, stacking (`hstack`, `vstack`) **forces NumPy to allocate brand-new memory (Creates a Copy).**
Because NumPy arrays must be perfectly contiguous (continuous blocks in RAM), it cannot simply link two separate arrays together. It has to book a new, larger block of RAM and copy the data from the old arrays into the new one. 
*Heavy use of stacking in large `for` loops will crash your memory!*

In [4]:
# --- 2. STACKING ARRAYS ---
a4 = np.arange(12).reshape(3, 4)
a5 = np.arange(12, 24).reshape(3, 4)

# Horizontal Stacking (Adding columns side-by-side)
# RAM allocates space for (3, 12) matrix
h_stack = np.hstack((a4, a5, a4))
print("Horizontal Stacking:\n", h_stack)

# Vertical Stacking (Adding rows top-to-bottom)
# RAM allocates space for (9, 4) matrix
v_stack = np.vstack((a4, a5, a4))
print("\nVertical Stacking:\n", v_stack)

Horizontal Stacking:
 [[ 0  1  2  3 12 13 14 15  0  1  2  3]
 [ 4  5  6  7 16 17 18 19  4  5  6  7]
 [ 8  9 10 11 20 21 22 23  8  9 10 11]]

Vertical Stacking:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]
 [16 17 18 19]
 [20 21 22 23]
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


## 3. Splitting (The Pointer Trick)
**Memory Level Reality:** Splitting (`hsplit`, `vsplit`) is the opposite of stacking, but computationally, it is much smarter. 
When you split an array, NumPy **does not** create new arrays. It simply returns a list of **Views** (pointers) that restrict your access to specific chunks of the original array. This saves massive amounts of RAM.

In [5]:
# --- 3. SPLITTING ARRAYS ---
print("Original a4:\n", a4)

# Horizontal Splitting (Splits columns into 2 equal parts)
h_split_result = np.hsplit(a4, 2)
print("\nHorizontal Split (Part 1):\n", h_split_result[0])
print("Horizontal Split (Part 2):\n", h_split_result[1])

print("\nOriginal a5:\n", a5)

# Vertical Splitting (Splits rows into 3 equal parts)
v_split_result = np.vsplit(a5, 3)
print("\nVertical Split (Part 1):\n", v_split_result[0])
print("Vertical Split (Part 2):\n", v_split_result[1])

Original a4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Horizontal Split (Part 1):
 [[0 1]
 [4 5]
 [8 9]]
Horizontal Split (Part 2):
 [[ 2  3]
 [ 6  7]
 [10 11]]

Original a5:
 [[12 13 14 15]
 [16 17 18 19]
 [20 21 22 23]]

Vertical Split (Part 1):
 [[12 13 14 15]]
Vertical Split (Part 2):
 [[16 17 18 19]]
